In [ ]:
import os
import pandas as pd
from transformers import AutoModel, BartTokenizer, BartForSequenceClassification
from datasets import load_dataset
from sklearn.metrics import classification_report

In [ ]:
# Load the dataset
dataset = load_dataset("csv", data_files="intent_dataset.csv")
dataset = dataset["train"]

In [ ]:
intent_labels = [
    "Check Assigned Staff",
    "Get Period Activity",
    "Retrieve Period Details",
    "Get Class Information",
    "Get Student Details",
    "List Subjects For Student",
    "List Subjects For Class",
    "Retrieve Student Details",
    "Get Leave Details",
    "Get Event Details",
    "Get Event By Date"
]

label2id = {label: idx for idx, label in enumerate(intent_labels)}
id2label = {idx: label for label, idx in label2id.items()}

In [ ]:
dataset = dataset.map(lambda x: {
    "label": label2id[x["intent"]]
})

In [ ]:
tokenizer = BartTokenizer.from_pretrained("facebook/bart-large-mnli")

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=64
    )

dataset = dataset.map(tokenize, batched=True)
dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

In [ ]:
model = BartForSequenceClassification.from_pretrained(
    "facebook/bart-large-mnli",
    num_labels=len(intent_labels),
    label2id=label2id,
    id2label=id2label
)

In [ ]:
training_args = TrainingArguments(
    output_dir="./bart-intent",
    evaluation_strategy="no",
    per_device_train_batch_size=8,
    num_train_epochs=10,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_steps=10,
    save_strategy="epoch",
    report_to="none"
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    tokenizer=tokenizer
)

trainer.train()

In [ ]:
model.save_pretrained("./bart-intent")
tokenizer.save_pretrained("./bart-intent")

In [ ]:
import torch

def predict_intent(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True)
    with torch.no_grad():
        outputs = model(**inputs)
    predicted_id = outputs.logits.argmax(dim=1).item()
    return id2label[predicted_id]

predict_intent("Who is teaching Maths for grade 7?")

## Refer this link
[Intent Classification](https://huggingface.co/facebook/bart-large-mnli)

[Entity Extraction](https://medium.com/@zilliz_learn/gliner-generalist-model-for-named-entity-recognition-using-bidirectional-transformer-ed65165a4877)